# Machine Learning Workflow

This document outlines the workflow for training, fitting, and evaluating machine learning models. Examples of utilising SHAP are also included. Since the workflow is essentially the same for all models, we provide only one workflow here, but we will include hyperparameter grids for all the models we use.  

We suggest conducting any experiment in a separate conda environment. 

## Libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm as cm
import os
from datetime import datetime, timedelta
import seaborn as sns
import xarray as xr
import dask.dataframe as dd
from dask import delayed
from functools import lru_cache
from tqdm import tqdm  
import warnings

In [ ]:
from shapely.geometry import Point, Polygon
import contextily as ctx
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import folium

In [ ]:
from scipy.spatial import cKDTree
import rasterio
from rasterio.transform import from_origin
from osgeo import gdal, osr
import regionmask

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.optimize import fmin_l_bfgs_b
from xgboost import XGBRegressor
import xgboost
import joblib
from lightgbm import LGBMRegressor
from lightgbm import early_stopping
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, DotProduct, WhiteKernel, ConstantKernel
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, RidgeCV, LassoCV, ElasticNetCV
from sklearn.neural_network import MLPRegressor

In [ ]:
import shap
from sklearn.inspection import PartialDependenceDisplay

## Prepare Dataframe/Feature Table

All-date DMS datapoints; Search Parquet with the unique dates from DMS; Annotation with interpolation  
  
NaN Value (fill-on):   

PAR - NaN    
CHL - -999  
NO3 - 9.96921e+36  
SST - -32768  
MLD - -32767  

First, merge the DMS data from two data sources: 

In [ ]:
dms_naames = pd.read_csv('processed_data/DMS_NAAMES.csv')
dms_pmel = pd.read_csv('processed_data/DMS_PMEL.csv')
dms_naames_copy = dms_naames.copy()
dms_pmel_copy = dms_pmel.copy()

In [ ]:
dms_naames_copy['date'] = pd.to_datetime(
    dms_naames_copy['date'],
    format='%Y-%m-%d %H:%M:%S.%f',  
    errors='raise'  
)

In [ ]:
dms_pmel_copy['time'] = pd.to_datetime(
    dms_pmel_copy['time'],
    format='%Y-%m-%dT%H:%M:%SZ',  # ISO 8601
    errors='raise'  
)

In [ ]:
dms_naames_copy = dms_naames_copy.rename(columns={'date': 'time'})
dms_naames_copy = dms_naames_copy.rename(columns={'lat': 'latitude', 'lon': 'longitude'})

In [ ]:
dms_combined = pd.concat([dms_naames_copy, dms_pmel_copy], ignore_index=True)
dms_combined['time'] = pd.to_datetime(dms_combined['time'], errors='raise')
print("dtype:", dms_combined['time'].dtype)
print("NaT:", dms_combined['time'].isna().sum())

In [ ]:
dms_combined.to_csv("processed_data/DMS_combined.csv", index=False)

In [ ]:
unique_dates = dms_combined['time'].dt.date.unique()  
print(f"Count: {len(unique_dates)}") # 399 days in all DMS observations
unique_dates_sorted = sorted(unique_dates)
dms_combined = dms_combined.sort_values(by='time').reset_index(drop=True)

### Env Data

My data paths:  
processed_data\CHL_20020704_20241130_parts  
processed_data\MLD_20020704_20210630_parts  
processed_data\NO3_20020704_20221231_parts  
processed_data\SST_CDSv2.1_parts  
processed_data\PAR

In [ ]:
# Define special values to replace with NaN for each variable
SPECIAL_VALUES = {
    'CHL': -999,
    'NO3': 9.96921e+36,
    'SST': -32768,
    'MLD': -32767
}

# Paths to Parquet files 
VAR_PATHS = {
    'CHL': 'processed_data/CHL_20020704_20241130_parts',
    'MLD': 'processed_data/MLD_20020704_20210630_parts',
    'NO3': 'processed_data/NO3_20020704_20221231_parts',
    'SST': 'processed_data/SST_CDSv2.1_parts',
    'PAR': 'processed_data/PAR'
}

#### Functions

Get file paths: 

In [ ]:
@lru_cache(maxsize=None)
def get_sst_file_date_mapping():
    base_path = VAR_PATHS['SST']
    mapping = {}
    for f in os.listdir(base_path):
        if f.startswith('part_') and f.endswith('.parquet'):
            file_path = os.path.join(base_path, f)
            try:
                df = dd.read_parquet(file_path, columns=['time']).compute()
                file_date = df['time'].iloc[0].date()
                mapping[file_date] = file_path
            except Exception as e:
                print(f"Error reading {file_path}: {str(e)}")
    return mapping

In [ ]:
def get_parquet_paths_for_dates(var_name, unique_dates):
    """
    Get paths to Parquet files covering the target dates.
    Args:
        var_name: Variable name (e.g., 'CHL')
        unique_dates: List of target dates (datetime.date)
    Returns:
        set: Unique Parquet file paths needed for these dates.
    """
    base_path = VAR_PATHS[var_name]
    file_paths = set()
    
    START_DATES = {
        'CHL': datetime(2002, 7, 4).date(),
        'MLD': datetime(2002, 7, 4).date(),
        'NO3': datetime(2002, 7, 4).date(),
        'SST': datetime(2002, 7, 1).date(),  
        'PAR': None  
    }

    for date in unique_dates:
        if var_name == 'PAR':
            # PAR files are named like PAR_20020704_20020902.parquet
            files = [f for f in os.listdir(base_path) if f.startswith('PAR_')]
            for f in files:
                start_str, end_str = f.split('_')[1:3]
                start_date = datetime.strptime(start_str, "%Y%m%d").date()
                end_date = datetime.strptime(end_str.split('.')[0], "%Y%m%d").date()
                if start_date <= date <= end_date:
                    file_paths.add(os.path.join(base_path, f))
                    print(f"Variable: {var_name}, Date: {date}, File: {file_paths}")
                    break
        elif var_name == 'SST': # parquet by day
            mapping = get_sst_file_date_mapping()
            if date in mapping:
                file_paths.add(mapping[date])
            else:
                print(f"Warning: No SST file found for date {date}")

        else:
            # Other vars use part_*.parquet (60-day chunks)
            start_date = START_DATES[var_name]
            days_since_start = (date - start_date).days
            if days_since_start < 0:
                print(f"Warning: Date {date} is before start date {start_date} for {var_name}")
                continue
            file_num = days_since_start // 60
            file_paths.add(os.path.join(base_path, f"part_{file_num:05d}.parquet"))
    
            print(f"Variable: {var_name}, Date: {date}, File: {file_paths}")
    return sorted(file_paths)

Ocean Mask: 

In [ ]:
# check the mask
land = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land.mask(lons, lats) # type: ignore
land_mask.plot() # 0=land
land_mask

In [ ]:
def get_ocean_mask(lons, lats):
    """Generate ocean mask for given coordinates (True=ocean, False=land)"""
    land = regionmask.defined_regions.natural_earth_v5_0_0.land_110
    # Get unique lons and lats 
    unique_lons = np.unique(lons)
    unique_lats = np.unique(lats)
    # Flatten the grid 
    grid_mask = np.isnan(land.mask(unique_lons, unique_lats).values)
    return grid_mask.flatten()

In [ ]:
# Test, should be True
get_ocean_mask(np.array([-20]), np.array([30]))

In [ ]:
# Test, should be false
get_ocean_mask(np.array([0]), np.array([20]))

Read, mask and interpolation

In [ ]:
@delayed
def load_var_for_date(var_name, target_date):
    """Load variable data for the specified date."""
    try:
        file_path = get_parquet_paths_for_dates(var_name, [target_date])[0]
        ddf = dd.read_parquet(file_path, columns=['time', 'latitude', 'longitude', var_name])
        ddf = ddf[ddf['time'].dt.date == target_date]
        if var_name in SPECIAL_VALUES:
            ddf[var_name] = ddf[var_name].replace(SPECIAL_VALUES[var_name], np.nan)
        return ddf
    except Exception:
        return None

In [ ]:
def temporal_interpolate(missing_data, var_name, prev_days, next_days):
    """
    Temporal interpolation with a 7-day window (3 days before + missing day + 3 days after).
    Handles continuous missing values and prioritizes nearest valid data for interpolation.
    
    Args:
        missing_data (pd.DataFrame): Data for the missing day (may be empty).
        var_name (str): Target variable name (e.g., 'temperature') to interpolate.
        prev_days (list): List of DataFrames for the 3 preceding days (ordered [day-3, day-2, day-1]).
        next_days (list): List of DataFrames for the 3 following days (ordered [day+1, day+2, day+3]).
    
    Returns:
        pd.DataFrame or None: Interpolated data for the missing day, or None if interpolation is impossible.
    """
    # Check if all 3 preceding days AND all 3 following days are empty/None
    all_prev_empty = all(data is None or data.empty for data in prev_days)
    all_next_empty = all(data is None or data.empty for data in next_days)
    
    if all_prev_empty and all_next_empty:
        print("Warning: All 7 consecutive days (3 before + missing + 3 after) are empty. Cannot interpolate. ")
        return None
    
    # Initialize missing_data structure if empty
    if missing_data.empty:
        # Try to copy structure from the nearest valid day in preceding days (reverse search) 优先用前 3 天中最近的非空数据初始化结构
        for prev_data in reversed(prev_days):
            if prev_data is not None and not prev_data.empty:
                missing_data = prev_data.copy()
                missing_data[var_name] = np.nan  # Mark target variable as missing
                break
        else:
            # If all preceding days are empty, try the nearest valid day in following days 如果前 3 天全空，尝试用后 3 天
            for next_data in next_days:
                if next_data is not None and not next_data.empty:
                    missing_data = next_data.copy()
                    missing_data[var_name] = np.nan
                    break
            else:
                return None  # No valid data in the entire 7-day window (edge case)
    
    # Find the nearest valid preceding value (search from day-1 to day-3) 查找最近的非空前驱数据（从 day-1 到 day-3 反向搜索）
    prev_valid = None
    for prev_data in reversed(prev_days):
        if prev_data is not None and not prev_data.empty:
            prev_valid = prev_data[var_name]
            break
    
    # Find the nearest valid following value (search from day+1 to day+3) 查找最近的非空后继数据（从 day+1 到 day+3 正向搜索）
    next_valid = None
    for next_data in next_days:
        if next_data is not None and not next_data.empty:
            next_valid = next_data[var_name]
            break
    
    # Perform interpolation
    if prev_valid is not None and next_valid is not None:
        missing_data[var_name] = (prev_valid + next_valid) / 2  # Linear interpolation
    elif prev_valid is not None:
        missing_data[var_name] = prev_valid  # Fill with nearest preceding value
    elif next_valid is not None:
        missing_data[var_name] = next_valid  # Fill with nearest following value
    else:
        return None  # Should not reach here due to earlier checks
    
    return missing_data

In [ ]:
def filter_ocean_points(df, mask):
    """Filter ocean points using precomputed mask."""
    return df[mask]

In [ ]:
@delayed
def interpolate_to_dms(ocean_df, var_name, dms_points, min_valid=4, max_k=16):
    """
    Robust IDW interpolation that handles NaN values near coastlines.
    
    Args:
        ocean_df (pd.DataFrame): Gridded data with columns ['latitude', 'longitude', var_name].
        var_name (str): Target variable name.
        dms_points (pd.DataFrame): DMS points to interpolate to.
        min_valid (int): Minimum required non-NaN neighbors (default: 4).
        max_k (int): Maximum neighbors to search (default: 16).
    
    Returns:
        list: Interpolated values at DMS points (NaN if insufficient valid neighbors).
    """
    # Build KDTree for spatial query
    coords = np.c_[ocean_df['latitude'], ocean_df['longitude']]
    values = ocean_df[var_name].values
    tree = cKDTree(coords)
    
    interpolated = []
    for _, row in dms_points.iterrows():
        target = [row['latitude'], row['longitude']]
        
        # Step 1: Query up to max_k neighbors
        k = min(max_k, len(ocean_df))  # Avoid exceeding available points
        dists, idxs = tree.query(target, k=k)
        
        # Step 2: Find the first 4 non-NaN neighbors
        valid_mask = ~np.isnan(values[idxs])
        valid_dists = dists[valid_mask][:min_valid]
        valid_vals = values[idxs][valid_mask][:min_valid]
        
        # Case 1: Enough valid neighbors -> IDW with 4 neighbours
        if len(valid_vals) >= min_valid:
            weights = 1 / (valid_dists[:4] + 1e-12)  # IDW
            interpolated_val = np.sum(weights * valid_vals[:4]) / np.sum(weights)
        
        # Case 2: Fewer than min_valid neighbors -> use available non-NaN value
        elif len(valid_vals) > 0:
            weights = 1 / (valid_dists + 1e-12)
            interpolated_val = np.sum(weights * valid_vals) / np.sum(weights)
            print(f"Warning: Only {len(valid_vals)} valid neighbors for DMS point {row.name}")
        
        # Case 3: All neighbors NaN -> propagate NaN
        else:
            interpolated_val = np.nan
            print(f"Warning: DMS point {row.name} is far inland with no valid neighbours.")
        interpolated.append(interpolated_val)
    
    return interpolated

In [ ]:
def process_daily_data(var_name, target_date, dms_points):
    """Process data for a single day."""
    df = load_var_for_date(var_name, target_date).compute()
    df = df.compute()  # Convert Dask DataFrame to Pandas DataFrame
    print(f"DataFrame shape for {var_name}: {df.shape}")
    if df is None or df.empty or df[var_name].isna().all():
        print(f"All NaN for {var_name} on {target_date}, trying interpolation...")
        # Load data for 3 preceding and 3 following days
        prev_days = [
            load_var_for_date(var_name, target_date - pd.Timedelta(days=i)).compute()
            for i in range(3, 0, -1)  # [day-3, day-2, day-1]
        ]
        next_days = [
            load_var_for_date(var_name, target_date + pd.Timedelta(days=i)).compute()
            for i in range(1, 4)  # [day+1, day+2, day+3]
        ]
        # Debug print shapes
        for i, data in enumerate(prev_days, start=1):
            print(f"Prev day-{i} shape: {data.shape if data is not None else 'None'}")
        for i, data in enumerate(next_days, start=1):
            print(f"Next day+{i} shape: {data.shape if data is not None else 'None'}")

        # Temporally interpolate using the 7-day window
        df = temporal_interpolate(
            missing_data=df if df is not None else pd.DataFrame(),
            var_name=var_name,
            prev_days=prev_days,
            next_days=next_days
        )
        print(f"Interpolated DataFrame shape: {df.shape if df is not None else 'None'}")
        
        if df is None or df.empty:
            print(f"Failed to interpolate {var_name} for {target_date} (7-day window unavailable)")
            return None

    # Get feature value for DMS points
    if df is not None and not df.empty:
        mask = get_ocean_mask(df['longitude'].values, df['latitude'].values)
        ocean_df = filter_ocean_points(df, mask)
        print(f"Filtered ocean points for {var_name} on {target_date}")
        return interpolate_to_dms(ocean_df, var_name, dms_points).compute()
    return None

#### Test functions

In [ ]:
target_date = pd.to_datetime("2004-01-27").date()
dms_daily = dms_combined[dms_combined['time'].dt.date == target_date]

results = {}
for var in ['CHL', 'MLD', 'NO3', 'SST', 'PAR']:
    interpolated = process_daily_data(var, target_date, dms_daily)
    if interpolated:
        results[var] = interpolated

clean_df = dms_daily.copy()
for var, values in results.items():
    clean_df[var] = values
clean_df = clean_df.drop(columns=['year', 'date'])
clean_df

In [ ]:
sst_paths = get_parquet_paths_for_dates('SST', [target_date])
sst_paths

test_df = dd.read_parquet(sst_paths[0]).compute()
test_df

### Get features annotated

In [ ]:
final_results = []

# Process by date with progress bar
for target_date in tqdm(unique_dates_sorted, desc="Processing dates"):
    print(f"\nProcessing date: {target_date}")
    
    # 1. Fetch and prepare DMS data for the target date
    dms_daily = dms_combined[
        dms_combined['time'].dt.date == target_date
    ].copy().reset_index(drop=True)
    
    if len(dms_daily) == 0:
        print(f"No DMS data for {target_date}")
        continue
    
    # 2. Process each variable with error handling
    daily_results = {}
    for var in ['CHL', 'MLD', 'NO3', 'SST', 'PAR']:
        try:
            # Process with ocean masking and interpolation
            interpolated = process_daily_data(var, target_date, dms_daily)
            
            if interpolated is not None:
                # Validate shape matches DMS points
                if len(interpolated) == len(dms_daily):
                    daily_results[var] = interpolated
                else:
                    print(f"Shape mismatch for {var} on {target_date}: "
                          f"Expected {len(dms_daily)}, got {len(interpolated)}")
        except Exception as e:
            print(f"Error processing {var} for {target_date}: {str(e)}")
            continue
    
    # 3. Compile and store results
    if daily_results:
        # Create a copy of the daily DMS data
        result_df = dms_daily.copy()
        
        # Add interpolated variables
        for var, values in daily_results.items():
            result_df[var] = values
        
        final_results.append(result_df)
        
        print(f"Successfully processed {len(daily_results)} variables for {target_date}")
    else:
        print(f"No variables processed for {target_date}")

# Combine all results
if final_results:
    final_df = pd.concat(final_results, ignore_index=True)
    print(f"\nProcessing complete. Final dataset contains {len(final_df)} records.")
else:
    print("\nNo data processed.")
    final_df = pd.DataFrame()

# Save final results
final_df.to_csv("processed_data/dms_with_env_vars_all.csv", index=False)

In [ ]:
env_vars = ['CHL', 'MLD', 'NO3', 'SST', 'PAR']

## Split set

In [ ]:
X = final_df[['CHL', 'MLD', 'NO3', 'SST', 'PAR']]
y = final_df['DMS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Remember to save the scaler for future generalisation
joblib.dump(scaler, r"C:\Your path\standard_scaler.joblib")

In [ ]:
feature_names = X_train.columns

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test.index)
X_scaled = pd.concat([X_train_scaled_df, X_test_scaled_df]).sort_index()

## XGB with workflow

In [ ]:
# Grid search for XGBoost, with 5-fold cross-validation and parallel processing
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1]
}

xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',  
    cv=5,  
    verbose=3,
    n_jobs=-1  
)

grid_search.fit(X_train_scaled, y_train)

# Here are the parameter set and performance metrics on the train set
print("Best params: ", grid_search.best_params_)
print("Best MSE score for train set: ", -grid_search.best_score_)  
print("Best model: ", grid_search.best_estimator_)

y_train_pred = grid_search.best_estimator_.predict(X_train_scaled)
r2_train = r2_score(y_train, y_train_pred)
print(f"Train R^2: {r2_train:.4f}")

In [ ]:
# After training, evaluate the performance on the test set
best_xgb_model = grid_search.best_estimator_
y_pred = best_xgb_model.predict(X_test_scaled)

xgb_mse = mean_squared_error(y_test, y_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
xgb_r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {xgb_mse}")
print(f"Test RMSE: {xgb_rmse}")
print(f"Test R^2: {xgb_r2}")

In [ ]:
# For plotting the prediction performance: 
# Record the predictions for the entire dataset (train + test) in our original dataframe
xgb_pred = best_xgb_model.predict(X_scaled)
final_df["xgb_pred"] = xgb_pred

# Remember to save the model for future use
model_save_path = r"C:\Your Path\best_xgb_model.joblib"
joblib.dump(best_xgb_model, model_save_path)

# Plot observed vs predicted DMS for the entire dataset
plt.figure(figsize=(10, 6))
plt.scatter(final_df['DMS'], final_df['xgb_pred'], alpha=0.5)
plt.plot([final_df['DMS'].min(), final_df['DMS'].max()], [final_df['DMS'].min(), final_df['DMS'].max()], 'r--')
plt.xlabel('Observed DMS')
plt.ylabel('Predicted DMS')
plt.title('Observed vs Predicted DMS (XGB)')
plt.savefig(r"C:\Your Path\XGB5fGridSearchObsVsPred.png", dpi=600)
plt.show()

## GPR with workflow

In [ ]:
# Resample 6000 samples to save memory
X_sample, y_sample = resample(
    X_train_scaled, y_train, 
    n_samples=6000, 
    random_state=42
)

In [ ]:
# Define kernels for Gaussian Process Regression
kernel_rbf = ConstantKernel(1.0, (1e-2, 1e2)) * RBF(length_scale=1.0)
kernel_matern = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(length_scale=1.0, nu=1.5)
kernel_rq = ConstantKernel(1.0, (1e-2, 1e2)) * RationalQuadratic(length_scale=1.0, alpha=1.0)

param_grid = {
    'kernel': [kernel_rbf, kernel_matern, kernel_rq],  
    'alpha': [1e-5, 1e-2],                 
    'n_restarts_optimizer': [0],         
    'normalize_y': [True],                  
}

In [ ]:
gpr = GaussianProcessRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=gpr,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=3,
    n_jobs=-1
)

grid_search.fit(X_sample, y_sample)
print("Best params: ", grid_search.best_params_)
print("Best MSE score for train set: ", -grid_search.best_score_)
print("Best model: ", grid_search.best_estimator_)
y_train_pred = grid_search.best_estimator_.predict(X_train_scaled)
r2_train = r2_score(y_train, y_train_pred)
print(f"Train R^2: {r2_train:.4f}")

In [ ]:
best_gpr_model = grid_search.best_estimator_
y_pred = best_gpr_model.predict(X_test_scaled)

gpr_mse = mean_squared_error(y_test, y_pred)
gpr_rmse = np.sqrt(gpr_mse)
gpr_r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {gpr_mse}")
print(f"Test RMSE: {gpr_rmse}")
print(f"Test R^2: {gpr_r2}")

model_save_path = r"C:\Your path\best_gpr_model.joblib"
joblib.dump(best_gpr_model, model_save_path)

In [ ]:
gpr_pred_2 = best_gpr_model.predict(X_scaled)
final_df["gpr_pred_2"] = gpr_pred_2

plt.figure(figsize=(10, 6))
plt.scatter(final_df['DMS'], final_df['gpr_pred_2'], alpha=0.5)
plt.plot([final_df['DMS'].min(), final_df['DMS'].max()], [final_df['DMS'].min(), final_df['DMS'].max()], 'r--')
plt.xlabel('Observed DMS')
plt.ylabel('Predicted DMS')
plt.title('Observed vs Predicted DMS (GPR)')
plt.savefig(r"C:\Your path\GPR5fGridSearchObsvsPred.png", dpi=600)
plt.show()

## RFR

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)

grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, 
                              cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search_rf.fit(X_train_scaled, y_train)

print(f"Best params for Random Forest: {grid_search_rf.best_params_}")
print(f"Best CV MSE for Random Forest: {-grid_search_rf.best_score_}")
print("Best model: ", grid_search_rf.best_estimator_)

y_train_pred = grid_search_rf.best_estimator_.predict(X_train_scaled)
r2_train = r2_score(y_train, y_train_pred)
print(f"Train R^2: {r2_train:.4f}")

Same workflow for evaluation on test set. Will not repeat in demonstration here. 

## SVR

In [ ]:
param_grid_svr = {
    'kernel': ['rbf'],
    'C': [10, 50, 100, 150],        
    'epsilon': [0.2, 0.3, 0.5, 0.7],  
    'gamma': ['scale', 'auto', 0.1, 1, 2]  
}

svr = SVR()

grid_search_svr = GridSearchCV(estimator=svr, param_grid=param_grid_svr, 
                              cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search_svr.fit(X_train_scaled, y_train)

print(f"Best params: {grid_search_svr.best_params_}")
print(f"Best CV MSE: {-grid_search_svr.best_score_}")
print("Best model: ", grid_search_svr.best_estimator_)

## MLP

In [ ]:
param_dist_mlp = {
    'hidden_layer_sizes': [(100, 50, 20), (128, 64, 32), (150, 100, 50), (100, 80, 60, 40), (100, 100, 50, 20), (256, 128, 64, 32), (128, 64, 32, 16)],  # Different architectures
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001, 0.01], # L2 regularisation params
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [1000, 2000],
}

mlp = MLPRegressor(early_stopping=True, validation_fraction=0.1, random_state=42)

In [ ]:
# grid search
grid_search_mlp = GridSearchCV(
    estimator=mlp,
    param_grid=param_dist_mlp,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    n_jobs=-1
)
grid_search_mlp.fit(X_train_scaled, y_train)
print(f"Best parameters from Grid Search: {grid_search_mlp.best_params_}")
print(f"Best MSE from Grid Search: {-grid_search_mlp.best_score_}")
print("Best model: ", grid_search_mlp.best_estimator_)

## LightGBM

In [ ]:
param_grid_lgb_tuned = {
    'num_leaves': [63, 80],
    'max_depth': [12, 15, 20],
    'min_child_samples': [20, 30],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0.001, 0.01],
    'reg_lambda': [0.05, 0.1, 0.2],
    'learning_rate': [0.03, 0.05],
    'n_estimators': [400, 500]
}

In [ ]:
lgb_model = LGBMRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=lgb_model,
    param_grid=param_grid_lgb_tuned,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV MSE:", -grid_search.best_score_)

After get the best param set, split a separate set from train set, and use Early Stopping to fit the model

In [ ]:
best_lgb_model = LGBMRegressor(**grid_search.best_params_, random_state=42)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

best_lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[early_stopping(stopping_rounds=50)]
)

In [ ]:
# Compare overfitting
model_path = r"C:\Your path\best_lgb_model.joblib"
loaded_model = joblib.load(model_path)

y_train_pred = loaded_model.predict(X_train_scaled)
r2_train = r2_score(y_train, y_train_pred)
print(f"Train R^2: {r2_train:.4f}")

y_test_pred = loaded_model.predict(X_test_scaled)
r2_test = r2_score(y_test, y_test_pred)
print(f"Test R^2: {r2_test:.4f}")

### Example with SHAP Analysis

SHAP in other models basically follows the same process: 

In [ ]:
explainer = shap.Explainer(loaded_model, X_train_scaled)
shap_values = explainer(X_train_scaled, check_additivity=False)
shap_values

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, feature_names=X_train.columns, show = True)
plt.savefig(r"C:\Your path\LGBMSHAP.png", dpi=600)

In [ ]:
shap.plots.waterfall(shap_values[0])
plt.savefig(r"C:\Your path\LGBMSHAPWaterfall.png", dpi=600)

#### Partial Dependency Plot

In [ ]:
shap_values = shap_values[:, :-1]

feature_names = X_train.columns
shap_val_df = pd.DataFrame(shap_values.values, columns=feature_names)
mean_abs_shap = shap_val_df.abs().mean().sort_values(ascending=False)
top_features = mean_abs_shap.index.tolist()
top_features

In [ ]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
disp = PartialDependenceDisplay.from_estimator(
    loaded_model,
    X=X_train_scaled_df,
    features=top_features,  
    ax=ax
)

fig.subplots_adjust(hspace=1)
fig.tight_layout()
plt.savefig(r"C:\Your path\LGBMPDP.png", dpi=600)
plt.show()

## Stacking

Fitted models as base learners: 

In [ ]:
best_gpr_model = joblib.load(r"C:\Your path\best_gpr_model.joblib")
best_xgb_model = joblib.load(r"C:\Your path\best_xgb_model.joblib")
best_rf_model = joblib.load(r"C:\Your path\best_rf_model.joblib")
best_svr_model = joblib.load(r"C:\Your path\best_svr_model.joblib")
best_mlp_model = joblib.load(r"C:\Your path\best_mlp_model.joblib")
best_lgb_model = joblib.load(r"C:\Your path\best_lgb_model.joblib")


In [ ]:
base_learners = [
    ('gpr', best_gpr_model),
    ('xgb', best_xgb_model),
    ('rf', best_rf_model),
    ('lgb', best_lgb_model),
    ('svr', best_svr_model),
    ('mlp', best_mlp_model)
]

### Test & Compare

According to results from our preliminary study, we test Ridge Regression first: 

For ridge regression: repeat the workflow first and check performance

In [ ]:
ridge_alphas = [0.1, 1.0, 10.0, 50.0, 100.0]

meta_learner = RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5)

In [ ]:
stacked_model = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=True,  
    n_jobs=-1,
    cv=5
)

stacked_model.fit(X_train_scaled, y_train)

print("Best Ridge alpha for meta-learner: ", stacked_model.final_estimator_.alpha_)
print("Stacked model coefficients: ", stacked_model.final_estimator_.coef_)
print("Stacked model intercept: ", stacked_model.final_estimator_.intercept_)
print("Best CV R2 for Stacked model: ", -stacked_model.score(X_train_scaled, y_train))

mse_scores = cross_val_score(
    stacked_model, X_train_scaled, y_train,
    scoring='neg_mean_squared_error',
    cv=5
)

print("Mean CV MSE:", -np.mean(mse_scores))

In [ ]:
y_pred_stack = stacked_model.predict(X_test_scaled)

mse_stack = mean_squared_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mse_stack)
r2_stack = r2_score(y_test, y_pred_stack)

print(f"Stacked Model - Test MSE: {mse_stack}")
print(f"Stacked Model - Test RMSE: {rmse_stack}")
print(f"Stacked Model - Test R^2: {r2_stack}")

model_save_path = r"C:\Your path\best_stacking_model.joblib"
joblib.dump(stacked_model, model_save_path)

In [ ]:
# Plot
stack_pred = stacked_model.predict(X_scaled)
final_df["stack_pred"] = stack_pred

plt.figure(figsize=(10, 6))
plt.scatter(final_df['DMS'], final_df['stack_pred'], alpha=0.5)
plt.plot([final_df['DMS'].min(), final_df['DMS'].max()],
         [final_df['DMS'].min(), final_df['DMS'].max()], 'r--')
plt.xlabel('Observed DMS')
plt.ylabel('Predicted DMS')
plt.title('Observed vs Predicted DMS (Stacking)')
plt.savefig(r"C:\Your path\Stack5fGridSearchObsvPred.png", dpi=600)
plt.show()

Check alpha: 

In [ ]:
# See how alpha affects the model
base_predictions_train = pd.DataFrame({
    name: model.predict(X_train_scaled)
    for name, model in base_learners
})

base_predictions_test = pd.DataFrame({
    name: model.predict(X_test_scaled)
    for name, model in base_learners
})

In [ ]:
ridge_alphas = [0.1, 1.0, 10.0, 50.0, 100.0]
ridge_cv = RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(base_predictions_train, y_train)

In [ ]:
mean_mse_per_alpha = []
for alpha in ridge_alphas:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(model, base_predictions_train, y_train, scoring='neg_mean_squared_error', cv=5)
    mean_mse_per_alpha.append(-scores.mean())


In [ ]:
# mean MSE with various ridge alpha values
plt.figure(figsize=(10, 6))
plt.plot(ridge_alphas, mean_mse_per_alpha, marker='o')
plt.xscale('log')
plt.xlabel('Ridge Alpha')
plt.ylabel('Mean CV MSE')
plt.title('Ridge Alpha vs CV MSE (Meta-Learner)')
plt.grid(True)
plt.savefig(r"C:\Your path\Stack5fCVMSEvsAlpha.png", dpi=600)
plt.show()

Try larger value range of alpha: 

In [ ]:
## Rerun stacking Ridge with larger alpha range
ridge_alphas = np.logspace(-2, 4, 20)

base_learners = [
    ('gpr', best_gpr_model),
    ('xgb', best_xgb_model),
    ('rf', best_rf_model),
    ('lgb', best_lgb_model),
    ('svr', best_svr_model),
    ('mlp', best_mlp_model)
]
meta_learner = RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5)

stacked_model_2_new_alpha = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=True,  
    n_jobs=-1,
    cv=5
)

stacked_model_2_new_alpha.fit(X_train_scaled, y_train)

print("Best Ridge alpha for meta-learner: ", stacked_model_2_new_alpha.final_estimator_.alpha_)
print("Stacked model coefficients: ", stacked_model_2_new_alpha.final_estimator_.coef_)
print("Stacked model intercept: ", stacked_model_2_new_alpha.final_estimator_.intercept_)
print("Best CV R2 for Stacked model: ", stacked_model_2_new_alpha.score(X_train_scaled, y_train))

mse_scores = cross_val_score(
    stacked_model_2_new_alpha, X_train_scaled, y_train,
    scoring='neg_mean_squared_error',
    cv=5
)

print("Mean CV MSE:", -np.mean(mse_scores))

y_pred_stack_new_alpha = stacked_model_2_new_alpha.predict(X_test_scaled)

mse_stack_2_new = mean_squared_error(y_test, y_pred_stack_new_alpha)
rmse_stack_2_new = np.sqrt(mse_stack_2_new)
r2_stack_2_new = r2_score(y_test, y_pred_stack_new_alpha)

print(f"Stacked Model - Test MSE: {mse_stack_2_new}")
print(f"Stacked Model - Test RMSE: {rmse_stack_2_new}")
print(f"Stacked Model - Test R^2: {r2_stack_2_new}")

model_save_path = r"C:\Your path\best_stacking_model_2_new_alpha.joblib"
joblib.dump(stacked_model_2_new_alpha, model_save_path)

As ridge regression still has large overfitting issue, let's start looking at other models as meta learner: Lasso, ElasticNet, with ridgeCV as a controlled set

In [ ]:
ridge_alphas = np.logspace(-2, 3, 10)

X_stacking_train = np.hstack([base_predictions_train, X_train_scaled])
X_stacking_test = np.hstack([base_predictions_test, X_test_scaled])

scaler = StandardScaler()
X_stacking_train_scaled = scaler.fit_transform(X_stacking_train)
X_stacking_test_scaled = scaler.transform(X_stacking_test)

In [ ]:
ridge = RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5)
ridge.fit(X_stacking_train_scaled, y_train)
ridge_preds = ridge.predict(X_stacking_test_scaled)

print("RidgeCV Results")
print("Best alpha:", ridge.alpha_)
print("CV R2:", ridge.score(X_stacking_train_scaled, y_train))
print("Test R2:", r2_score(y_test, ridge_preds))
print("Test MSE:", mean_squared_error(y_test, ridge_preds))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, ridge_preds)))
print("Coefficients:", ridge.coef_, "\n")

In [ ]:
lasso = LassoCV(alphas=ridge_alphas, cv=5)
lasso.fit(X_stacking_train_scaled, y_train)
lasso_preds = lasso.predict(X_stacking_test_scaled)

print("LassoCV Results")
print("Best alpha:", lasso.alpha_)
print("CV R2:", lasso.score(X_stacking_train_scaled, y_train))
print("Test R2:", r2_score(y_test, lasso_preds))
print("Test MSE:", mean_squared_error(y_test, lasso_preds))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, lasso_preds)))
print("Coefficients:", lasso.coef_, "\n")

In [ ]:
elastic = ElasticNetCV(alphas=ridge_alphas, l1_ratio=[0.1, 0.5, 0.9], cv=5)
elastic.fit(X_stacking_train_scaled, y_train)
elastic_preds = elastic.predict(X_stacking_test_scaled)

print("ElasticNetCV Results")
print("Best alpha:", elastic.alpha_)
print("Best l1_ratio:", elastic.l1_ratio_)
print("CV R2:", elastic.score(X_stacking_train_scaled, y_train))
print("Test R2:", r2_score(y_test, elastic_preds))
print("Test MSE:", mean_squared_error(y_test, elastic_preds))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, elastic_preds)))
print("Coefficients:", elastic.coef_, "\n")

In [ ]:
# Plot comparison of meta-learners
models = ['Ridge', 'Lasso', 'ElasticNet']
r2_scores = [r2_score(y_test, ridge_preds), r2_score(y_test, lasso_preds), r2_score(y_test, elastic_preds)]
rmse_scores = [np.sqrt(mean_squared_error(y_test, ridge_preds)),
               np.sqrt(mean_squared_error(y_test, lasso_preds)), 
               np.sqrt(mean_squared_error(y_test, elastic_preds))]

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].bar(models, r2_scores, color='skyblue')
ax[0].set_title('Test R²')
ax[0].set_ylim(0.75, 0.78)
ax[1].bar(models, rmse_scores, color='salmon')
ax[1].set_title('Test RMSE')
ax[1].set_ylim(1.1, 1.25)
plt.suptitle("Meta-learners Performance Comparison")
plt.tight_layout()
plt.savefig(r"C:\Your path\RidgeLassoElasticNetCompare.png", dpi=600)
plt.show()

From the result, we found that ElasticNet has relatively the best generalisation ability. Then go on with it: 

### ElasticNet

In [ ]:
# StackingRegressor with ElasticNet
elastic_meta_learner = ElasticNetCV(
    alphas=np.logspace(-2, 3, 10),
    l1_ratio=[0.1, 0.5, 0.9],
    cv=5,
    random_state=42
)

stacked_model_2_elasticnet = StackingRegressor(
    estimators=base_learners,
    final_estimator=elastic_meta_learner,
    passthrough=True,
    n_jobs=-1,
    cv=5
)

stacked_model_2_elasticnet.fit(X_train_scaled, y_train)

Test if we should include original features or not: 

In [ ]:
# Passthrough = False
stacked_model_2_elasticnet = StackingRegressor(
    estimators=base_learners,
    final_estimator=elastic_meta_learner,
    passthrough=False,
    n_jobs=-1,
    cv=5
)

stacked_model_2_elasticnet.fit(X_train_scaled, y_train)

print("Best alpha for meta-learner: ", stacked_model_2_elasticnet.final_estimator_.alpha_)
print("Stacked model coefficients: ", stacked_model_2_elasticnet.final_estimator_.coef_)
print("Stacked model intercept: ", stacked_model_2_elasticnet.final_estimator_.intercept_)
print("Best CV R2 for Stacked model: ", stacked_model_2_elasticnet.score(X_train_scaled, y_train))

y_pred_stack_elasticnet = stacked_model_2_elasticnet.predict(X_test_scaled)

mse_stack_2_elasticnet = mean_squared_error(y_test, y_pred_stack_elasticnet)
rmse_stack_2_elasticnet = np.sqrt(mse_stack_2_elasticnet)
r2_stack_2_elasticnet = r2_score(y_test, y_pred_stack_elasticnet)

print(f"Stacked Model - Test MSE: {mse_stack_2_elasticnet:.4f}")
print(f"Stacked Model - Test RMSE: {rmse_stack_2_elasticnet:.4f}")
print(f"Stacked Model - Test R^2: {r2_stack_2_elasticnet:.4f}")

In [ ]:
# Results of Passthrough = True
print("Best Ridge alpha for meta-learner: ", stacked_model_2_elasticnet.final_estimator_.alpha_)
print("Stacked model coefficients: ", stacked_model_2_elasticnet.final_estimator_.coef_)
print("Stacked model intercept: ", stacked_model_2_elasticnet.final_estimator_.intercept_)
print("Best CV R2 for Stacked model: ", stacked_model_2_elasticnet.score(X_train_scaled, y_train))

mse_scores = cross_val_score(
    stacked_model_2_elasticnet, X_train_scaled, y_train,
    scoring='neg_mean_squared_error',
    cv=5
)

print("Mean CV MSE:", -np.mean(mse_scores))

y_pred_stack_elasticnet = stacked_model_2_elasticnet.predict(X_test_scaled)

mse_stack_2_elasticnet = mean_squared_error(y_test, y_pred_stack_elasticnet)
rmse_stack_2_elasticnet = np.sqrt(mse_stack_2_elasticnet)
r2_stack_2_elasticnet = r2_score(y_test, y_pred_stack_elasticnet)

print(f"Stacked Model - Test MSE: {mse_stack_2_elasticnet:.4f}")
print(f"Stacked Model - Test RMSE: {rmse_stack_2_elasticnet:.4f}")
print(f"Stacked Model - Test R^2: {r2_stack_2_elasticnet:.4f}")

Include original features has better performance. Therefore, save this model: 

In [ ]:
model_save_path = r"C:\Your path\best_stacking_model_elasticnet.joblib"
joblib.dump(stacked_model_2_elasticnet, model_save_path)

#### SHAP

In [ ]:
model_path = r"C:\Your path\best_stacking_model_elasticnet.joblib"
stacked_model_2_elasticnet = joblib.load(model_path)

The params of the model:  

StackingRegressor(cv=5,
                  estimators=[('gpr',
                               GaussianProcessRegressor(alpha=0.01,
                                                        kernel=1**2 * RationalQuadratic(alpha=1, length_scale=1),
                                                        normalize_y=True,
                                                        random_state=42)),
                              ('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=1.0, device=None,
                                            early_stopping_rounds=None,
                                            enable_categor...
                                            early_stopping=True,
                                            hidden_layer_sizes=(256, 128, 64,
                                                                32),
                                            max_iter=1000, random_state=42))],
                  final_estimator=ElasticNetCV(alphas=array([1.00000000e-02, 3.59381366e-02, 1.29154967e-01, 4.64158883e-01,
       1.66810054e+00, 5.99484250e+00, 2.15443469e+01, 7.74263683e+01,
       2.78255940e+02, 1.00000000e+03]),
                                               cv=5, l1_ratio=[0.1, 0.5, 0.9],
                                               random_state=42),
                  n_jobs=-1, passthrough=True)

In [ ]:
# As stacking not callable, we need to transform it first
X_meta = stacked_model_2_elasticnet.transform(X_train_scaled)

final_model = stacked_model_2_elasticnet.final_estimator_

In [ ]:
# SHAP for final estimator
explainer = shap.Explainer(final_model, X_meta)
shap_values = explainer(X_meta)

In [ ]:
original_feature_names = list(X_train.columns)

base_model_names = list(dict(stacked_model_2_elasticnet.named_estimators_).keys())
base_model_feature_names = [f"{name}_pred" for name in base_model_names]

full_feature_names = original_feature_names + base_model_feature_names

In [ ]:
plt.figure(figsize=(10, 6))
# Here we don't show the plot immediately to avoid display & saving issues
shap.summary_plot(shap_values, X_meta, feature_names=full_feature_names, show = False)
plt.tight_layout()
plt.savefig(r"C:\Your path\StackElasticSHAP.png", dpi=600)
plt.close()

In [ ]:
shap_values.feature_names = full_feature_names

In [ ]:
# replace the default names with our feature/model names
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, show = False)
plt.tight_layout()
plt.savefig(r"C:\Your path\StackElasticSHAPBar.png", dpi=600)
plt.close()

For other base learners: (please note that some are not available to use SHAP directly, and we need to use permutation explainer for those models)

In [ ]:
# SHAP for single models
save_dir = r"C:\Your path\base_learners"
os.makedirs(save_dir, exist_ok=True)

shap_dir = r"C:\Your path\shap"
os.makedirs(shap_dir, exist_ok=True)

In [ ]:
# Choose random sample for permutation model
sample_idx = np.random.choice(len(X_train_scaled), size=100, replace=False)
X_sampled = X_train_scaled[sample_idx]

In [ ]:
for name, model in base_learners:
    model_upper = name.upper()  # uppercase model name for display
    print(f"Processing model: {model_upper}")
    
    # Appropriate SHAP explainer
    if name in ['gpr', 'xgb']:
        continue
        '''elif name == 'xgb':
        shap_values = model.get_booster().predict(
            xgboost.DMatrix(X_train_scaled), pred_contribs=True
        )
        shap_values = shap_values[:, :-1]  # exclude bias term
        # summary plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_scaled, feature_names=X_train.columns, show=False)
        plt.title(f"SHAP summary for {model_upper}", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_shap_summary.png"), dpi=600)
        plt.close()'''
    elif name in ['rf', 'lgb']:
        explainer = shap.Explainer(model, X_train_scaled)
        shap_values = explainer(X_train_scaled, check_additivity=False)
        shap_values.feature_names = list(X_train.columns)
        # summary plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train_scaled, show=False)
        plt.title(f"SHAP summary for {model_upper}", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_shap_summary.png"), dpi=600)
        plt.close()
    
        # bar plot
        plt.figure(figsize=(10, 6))
        shap.plots.bar(shap_values, show=False)
        plt.title(f"SHAP bar for {model_upper}", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_shap_bar.png"), dpi=600)
        plt.close()
    else:
        # For mlp, gpr, and svr, use permutation explainer
        explainer = shap.Explainer(model.predict, X_sampled, algorithm="permutation")
        shap_values = explainer(X_sampled)
        shap_values.feature_names = list(X_train.columns)
        # summary plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_sampled, show=False)
        plt.title(f"SHAP summary for {model_upper}", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_shap_summary.png"), dpi=600)
        plt.close()
    
        # bar plot
        plt.figure(figsize=(10, 6))
        shap.plots.bar(shap_values, show=False)
        plt.title(f"SHAP bar for {model_upper}", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{name}_shap_bar.png"), dpi=600)
        plt.close()
    
    joblib.dump(shap_values, os.path.join(shap_dir, f"{model_upper}_shap_values.pkl"))
    joblib.dump(explainer, os.path.join(shap_dir, f"{model_upper}_explainer.pkl"))

In [ ]:
# Load SHAP values for all models
model_names = ['GPR', 'XGB', 'RF', 'LGB', 'SVR', 'MLP']
shap_values_dict = {}
for name in model_names:
    path = os.path.join(shap_dir, f"{name}_shap_values.pkl")
    if os.path.exists(path):
        shap_values_dict[name] = joblib.load(path)
    else:
        print(f"Missing SHAP file for {name}")

In [ ]:
shap_arrays = {}

for name, shap_value in shap_values_dict.items():
    if hasattr(shap_value, 'values'):  
        shap_arrays[name] = shap_value.values
    else:  
        shap_arrays[name] = shap_value

Consistency plotted as heat maps: 

In [ ]:
importance_matrix = pd.DataFrame({
    name: np.abs(arr).mean(axis=0) for name, arr in shap_arrays.items()
})

corr_matrix = importance_matrix.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("SHAP Feature Importance Consistency Across Models")
plt.savefig(os.path.join(save_dir, f"heatmap.png"), dpi=600)
plt.show()


In [ ]:
mean_abs_shap = {
    model: np.abs(shap_values).mean(axis=0)
    for model, shap_values in shap_arrays.items()
}

shap_df = pd.DataFrame(mean_abs_shap, index=feature_names).T

plt.figure(figsize=(6, 6))
sns.heatmap(shap_df, annot=True, fmt=".3f", cmap="coolwarm", cbar_kws={'label': 'Mean |SHAP|'})
plt.title("Mean Absolute SHAP Values per Feature Across Models")
plt.xlabel("Feature")
plt.ylabel("Model")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, f"shapvalue_heatmap.png"), dpi=600)
plt.show()

In [ ]:
# Partial Dependence Plots (PDP) for each model
def plot_pdp(model, model_name, X, shap_values, save_path):
    feature_names = X.columns
    shap_val_df = pd.DataFrame(shap_values.values, columns=feature_names)
    mean_abs_shap = shap_val_df.abs().mean().sort_values(ascending=False)
    top_features = mean_abs_shap.index.tolist()

    fig, ax = plt.subplots(figsize=(10, 9))
    PartialDependenceDisplay.from_estimator(
        model,
        X=X,
        features=top_features,  
        ax=ax
    )
    fig.subplots_adjust(hspace=1)
    fig.suptitle(f"PDP for {model_name.upper()}", fontsize=16)
    fig.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

In [ ]:
plot_pdp(best_rf_model, 'RF', X_train_scaled_df, shap_values_dict['RF'], os.path.join(save_dir, 'RF_PDP.png'))

### Test on Simplified Combination

As tree-based base-learners have better performance than others (GPR, MLP, SVR), we are also wondering if the performance can be improved when we only stack tree-based models. 

In [ ]:
base_learners = [
    ('xgb', best_xgb_model),
    ('rf', best_rf_model),
    ('lgb', best_lgb_model)
]

ridge_alphas = np.logspace(-3, 1, 100)  

In [ ]:
for use_passthrough in [False, True]:
    stacked_model_3 = StackingRegressor(
        estimators=base_learners,
        final_estimator=RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5),
        passthrough=use_passthrough,
        cv=5,
        n_jobs=-1
    )
    scores = cross_val_score(stacked_model_3, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
    print(f"Passthrough={use_passthrough} | Mean CV MSE: {-scores.mean():.4f}")

In [ ]:
stacked_model_3 = StackingRegressor(
        estimators=base_learners,
        final_estimator=RidgeCV(alphas=ridge_alphas, scoring='neg_mean_squared_error', cv=5),
        passthrough=True,
        cv=5,
        n_jobs=-1
    )

In [ ]:
stacked_model_3.fit(X_train_scaled, y_train)

print("Best Ridge alpha for meta-learner: ", stacked_model_3.final_estimator_.alpha_)
print("Stacked model coefficients: ", stacked_model_3.final_estimator_.coef_)
print("Stacked model intercept: ", stacked_model_3.final_estimator_.intercept_)
print("Best CV R2 for Stacked model: ", stacked_model_3.score(X_train_scaled, y_train))

y_pred_stack = stacked_model_3.predict(X_test_scaled)

mse_stack_3 = mean_squared_error(y_test, y_pred_stack)
rmse_stack_3 = np.sqrt(mse_stack_3)
r2_stack_3 = r2_score(y_test, y_pred_stack)

print(f"Stacked Model - Test MSE: {mse_stack_3}")
print(f"Stacked Model - Test RMSE: {rmse_stack_3}")
print(f"Stacked Model - Test R^2: {r2_stack_3}")

model_save_path = r"C:\Your path\best_stacking_model_3.joblib"
joblib.dump(stacked_model_3, model_save_path)

From our result, only stacking these three base-learners cannot improve either the test performance or the overfitting issue. Thus, let's keep our 6 base-learners with ElasticNet. 

In [ ]:
stack_pred_3 = stacked_model_3.predict(X_scaled)
final_df["stack_pred_3"] = stack_pred_3

plt.figure(figsize=(10, 6))
plt.scatter(final_df['DMS'], final_df['stack_pred_3'], alpha=0.5)
plt.plot([final_df['DMS'].min(), final_df['DMS'].max()],
         [final_df['DMS'].min(), final_df['DMS'].max()], 'r--')
plt.xlabel('Observed DMS')
plt.ylabel('Predicted DMS')
plt.title('Observed vs Predicted DMS (Stacking)')
plt.savefig(r"C:\Your path\Stack5fGridSearchObsvsPred_3.png", dpi=600)
plt.show()